# Connect to pawsey0411

Startup version validation. Catch dependency issues fail before scanning data

In [ ]:
import sys,json
from importlib import import_module
from importlib.metadata import version, PackageNotFoundError

REQUIRED_EXACT = {
    "fsspec": "2026.3.0",
    "s3fs": "2026.3.0",
    "kerchunk": "0.2.10",
    "virtualizarr": "2.6.0",
}
errors = []
report = {}

if sys.version_info <(3,12):
    errors.appennd("Python 3.12+ required for virtualizarr==2.6.0")

for pkg, expected_version in REQUIRED_EXACT.items():
    try:
        got_version= version(pkg)
        report[pkg]= got_version
        if got_version != expected_version:
            errors.append(f"{pkg} Version expected: {expected_version}, got: {got_version}")
    except PackageNotFoundError:
        errors.append(f"{pkg} missing")

for mod in ["fsspec", "s3fs", "xarray", "zarr", "kerchunk", "virtualizarr"]:
    try:
        import_module(mod)
    except Exception as exc:
        errors.append(f"import {mod} failed: {type(exc).__name__}: {exc}")

if report.get("fsspec") and report.get("s3fs") and report["fsspec"] != report["s3fs"]:
    errors.append(f"fsspec/s3fs mismatch: {report['fsspec']} vs {report['s3fs']}")

if errors:
    raise RuntimeError("Runtime readiness FAILED:\n- " + "\n- ".join(errors))

print("Runtime readiness PASSED")
print(json.dumps(report, indent=2, sort_keys=True))

In [ ]:
import socket
socket.gethostbyname("projects.pawsey.org.au")
socket.create_connection(("projects.pawsey.org.au", 443), timeout=10).close()
print("Endpoint reachable")

ENDPOINT = "https://projects.pawsey.org.au"
BUCKET = "weather"
SCOPE = "pawsey0411"

ACCESS_KEY = dbutils.secrets.get(scope=SCOPE, key="access_key")
SECRET_KEY = dbutils.secrets.get(scope=SCOPE, key="secret_key")

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

# Print visible buckets
try:
    print("Buckets visible:", [b["Name"] for b in s3.list_buckets().get("Buckets", [])])
except ClientError as e:
    print("list_buckets not allowed:", e.response.get("Error", {}))

# Equivalent to rclone lsd pawsey0411:weather
resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
print("Top-level pseudo-directories:", prefixes)

# Top-level objects sample
root_objects = [o["Key"] for o in resp.get("Contents", [])]
print("Top-level objects sample:", root_objects[:20])